In [85]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')


In [86]:
df = pd.read_csv('TRAIN.csv')

print("Shape:", df.shape)
print("Missing values :", df.isnull().sum().sum())
print("\nClass distribution:")
print(df['Class'].value_counts())

Shape: (43776, 48)
Missing values : 0

Class distribution:
Class
0    26465
1    17311
Name: count, dtype: int64


##  Define Feature Groups



We have observe that some of the features are the scaled version of other set of Features.
So we group feature in 5 groups
A -> F-01 to F-09
B -> F10 to  F-19
C -> f21 to f29
D -> f30 to f39
E -> f39 to f47

F20 variance to near 0 , it is working like a noise

In [87]:
grpA = [f'F{str(i).zfill(2)}' for i in range(1,  10)]
grpB = [f'F{str(i).zfill(2)}' for i in range(10, 20)]
grpC = [f'F{str(i).zfill(2)}' for i in range(21, 30)]
grpD = [f'F{str(i).zfill(2)}' for i in range(30, 39)]
grpE = [f'F{str(i).zfill(2)}' for i in range(39, 48)]

print("Groups defined.")
print(f"A={grpA}\nB={grpB}\nC={grpC}\nD={grpD}\nE={grpE}")

Groups defined.
A=['F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09']
B=['F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19']
C=['F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28', 'F29']
D=['F30', 'F31', 'F32', 'F33', 'F34', 'F35', 'F36', 'F37', 'F38']
E=['F39', 'F40', 'F41', 'F42', 'F43', 'F44', 'F45', 'F46', 'F47']




### Dropping the F20



In [88]:

df.drop(columns=['F20'], inplace=True)


### Replace Group C with C/A Ratio Features



We observe that group A and group C have to same significance with different scales.
So we will take the ration of those features. and we will drop the C group feature ,and A group Features will be as it is , to intack the physical significance of the ration .

 **ratio C/A**


In [89]:


print(f"{'Pair':<14} {'Correlation':>12} {'Median scale C/A':>18}")
print("-" * 46)
for fa, fc in zip(grpA, grpC):
    corr, _ = pearsonr(df[fa], df[fc])
    scale = (df[fc] / (df[fa] + 1e-9)).median()
    print(f"{fa} vs {fc}    {corr:>12.3f} {scale:>18.2f}x")

Pair            Correlation   Median scale C/A
----------------------------------------------
F01 vs F21           0.770               2.93x
F02 vs F22           0.940              48.12x
F03 vs F23           0.942              43.15x
F04 vs F24           0.975              14.62x
F05 vs F25           0.974               9.00x
F06 vs F26           0.996               4.58x
F07 vs F27           0.997               2.22x
F08 vs F28           0.984               0.95x
F09 vs F29           0.990               0.55x


In [90]:
# Ratio of grp a and c
for fa, fc in zip(grpA, grpC):
    df[f'ratio_{fc}_{fa}'] = df[fc] / (df[fa] + 1e-9)

df.drop(columns=grpC, inplace=True)


In [91]:

for fb, fd in zip(grpB, grpD):
    df[f'ratio_{fd}_{fb}'] = df[fd] / (df[fb].abs() + 1e-9)

# Drop original Group D
df.drop(columns=grpD, inplace=True)



In [92]:

from scipy.stats import pearsonr

sumA = df[grpA].sum(axis=1)
corr, _ = pearsonr(sumA, df['F19'])
print(f"Correlation of sum(A) with F19: {corr:.3f}")


slope = np.cov(sumA, df['F19'])[0,1] / np.var(sumA)
df['F19_residual'] = df['F19'] - slope * sumA

#Check it differs between classes
stat, p = mannwhitneyu(
    df['F19_residual'][df['Class']==0],
    df['F19_residual'][df['Class']==1],
    alternative='two-sided'
)
print(f"F19_residual p-value vs Class: {p:.2e}  (significant = good feature)")


Correlation of sum(A) with F19: 0.840
F19_residual p-value vs Class: 9.20e-45  (significant = good feature)


In [93]:
df['grpA_sum'] = df[grpA].sum(axis=1)
df['grpB_sum'] = df[grpB].sum(axis=1)

# Check discriminability
for col in ['grpA_sum', 'grpB_sum']:
    c0 = df[col][df['Class']==0].median()
    c1 = df[col][df['Class']==1].median()
    stat, p = mannwhitneyu(df[col][df['Class']==0], df[col][df['Class']==1], alternative='two-sided')
    print(f"{col}: Class0={c0:.3f}  Class1={c1:.3f}  ratio={c1/c0:.2f}x  p={p:.2e}")



grpA_sum: Class0=0.693  Class1=1.212  ratio=1.75x  p=0.00e+00
grpB_sum: Class0=31.088  Class1=34.323  ratio=1.10x  p=0.00e+00


In [94]:

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

print(f"Null values : {df.isnull().sum().sum()}")
print(f"Inf  values : {np.isinf(df.select_dtypes(include=np.number)).sum().sum()}")

Null values : 0
Inf  values : 0


In [95]:
df.to_csv('TRAIN_FE.csv', index=False)

df.head()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,ratio_F32_F12,ratio_F33_F13,ratio_F34_F14,ratio_F35_F15,ratio_F36_F16,ratio_F37_F17,ratio_F38_F18,F19_residual,grpA_sum,grpB_sum
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,1.015246,0.960946,0.837119,0.818194,0.739435,0.801747,0.745842,-0.252623,0.482497,32.077725
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,10.000095,2.753256,0.923644,0.984414,0.856845,0.761899,0.817288,-0.758267,0.975581,31.117146
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,3.306491,2.600483,0.866374,0.869999,0.882478,0.774873,0.749903,0.187816,1.670239,37.275633
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,266.947549,117.136034,95.478884,39.237070,5.164514,0.781215,0.835150,0.073217,0.929422,25.055210
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,1.417105,0.926842,0.975356,0.812273,0.954541,0.811071,1.006131,0.832870,0.633233,43.676510
